In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc mergedeep numpy qiskit-addon-aqc-tensor qiskit-addon-utils qiskit-ibm-catalog qiskit-serverless quimb scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Criar um template de função para simulação hamiltoniana
Este template encapsula um fluxo de trabalho para simular a evolução temporal de um estado inicial em relação a um Hamiltoniano baseado em spin definido pelo usuário e retorna um conjunto de valores esperados especificados usando o [addon AQC](./qiskit-addons-aqc).

Este template é estruturado como um padrão Qiskit com as seguintes etapas:

#### 1. Coleta de entradas e mapeamento do problema
Esta seção recebe como entrada o Hamiltoniano a simular, um estado inicial na forma de um `QuantumCircuit`, um conjunto de observáveis para estimar os valores esperados e uma especificação de opções para o addon AQC. Esta etapa valida que todos os dados de entrada necessários estão presentes e que estão no formato correto.

Os argumentos de entrada são então usados para construir os circuitos quânticos e operadores relevantes para o fluxo de trabalho. Um circuito alvo é criado e uma representação de estado produto matricial desse circuito é encontrada usando o addon AQC. Em seguida, um circuito ansatz é gerado e otimizado usando métodos de rede tensorial, produzindo um circuito final que executa o restante da evolução temporal.

#### 2. Preparar os circuitos gerados para execução
Os circuitos gerados pelo addon AQC são então transpilados para execução em um backend escolhido. Uma instância de [`EstimatorV2`](../api/qiskit-ibm-runtime/estimator-v2) é criada com um conjunto padrão de opções de mitigação de erros para gerenciar a execução do circuito.

#### 3. Execução
Por fim, o circuito ansatz é transpilado e executado em um QPU, coletando estimativas para todos os valores esperados especificados, que são retornados em um formato serializável para acesso pelo usuário.

## Escrever o template de função
Primeiro, escreva um template de função para simulação hamiltoniana que usa o [addon AQC-Tensor do Qiskit](/guides/qiskit-addons-aqc) para mapear a descrição do problema a um circuito de profundidade reduzida para execução em hardware.

Durante todo o processo, o código é salvo em `./source_files/template_hamiltonian_simulation.py`. Este arquivo é o template de função que você pode enviar e executar remotamente com o Qiskit Serverless.

In [1]:
# This cell is hidden from users, it just creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

### Coletar e validar as entradas
Comece obtendo as entradas para o template. Este exemplo tem entradas específicas do domínio relevantes para a simulação hamiltoniana (como o Hamiltoniano e o observável) e opções específicas de capacidade (como quanto você quer comprimir as camadas iniciais do circuito de Trotter usando AQC-Tensor, ou opções avançadas para ajuste fino da supressão e mitigação de erros além dos padrões que fazem parte deste exemplo).

In [2]:
%%writefile ./source_files/template_hamiltonian_simulation.py

from qiskit import QuantumCircuit
from qiskit_serverless import get_arguments, save_result


# Extract parameters from arguments
#
# Do this at the top of the program so it fails early if any required arguments are missing or invalid.

arguments = get_arguments()

dry_run = arguments.get("dry_run", False)
backend_name = arguments["backend_name"]

aqc_evolution_time = arguments["aqc_evolution_time"]
aqc_ansatz_num_trotter_steps = arguments["aqc_ansatz_num_trotter_steps"]
aqc_target_num_trotter_steps = arguments["aqc_target_num_trotter_steps"]

remainder_evolution_time = arguments["remainder_evolution_time"]
remainder_num_trotter_steps = arguments["remainder_num_trotter_steps"]

# Stop if this fidelity is achieved
aqc_stopping_fidelity = arguments.get("aqc_stopping_fidelity", 1.0)
# Stop after this number of iterations, even if stopping fidelity is not achieved
aqc_max_iterations = arguments.get("aqc_max_iterations", 500)

hamiltonian = arguments["hamiltonian"]
observable = arguments["observable"]
initial_state = arguments.get("initial_state", QuantumCircuit(hamiltonian.num_qubits))

Writing ./source_files/template_hamiltonian_simulation.py


In [3]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

import numpy as np
import json
from mergedeep import merge


# Configure `EstimatorOptions`, to control the parameters of the hardware experiment
#
# Set default options
estimator_default_options = {
    "resilience": {
        "measure_mitigation": True,
        "zne_mitigation": True,
        "zne": {
            "amplifier": "gate_folding",
            "noise_factors": [1, 2, 3],
            "extrapolated_noise_factors": list(np.linspace(0, 3, 31)),
            "extrapolator": ["exponential", "linear", "fallback"],
        },
        "measure_noise_learning": {
            "num_randomizations": 512,
            "shots_per_randomization": 512,
        },
    },
    "twirling": {
        "enable_gates": True,
        "enable_measure": True,
        "num_randomizations": 300,
        "shots_per_randomization": 100,
        "strategy": "active",
    },
}
# Merge with user-provided options
estimator_options = merge(
    arguments.get("estimator_options", {}), estimator_default_options
)

Appending to ./source_files/template_hamiltonian_simulation.py


When the function template is running, it is helpful to return information in the logs by using print statements, so that you can better evaluate the workload's progress. Following is a simple example of printing the `estimator_options` so  there is a record of the actual Estimator options used. There are many more similar examples throughout the program to report progress during execution, including the value of the objective function during the iterative component of AQC-Tensor, and the two-qubit depth of the final instruction set architecture (ISA) circuit intended for execution on hardware.

In [4]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

print("estimator_options =", json.dumps(estimator_options, indent=4))

Appending to ./source_files/template_hamiltonian_simulation.py


#### Validate the inputs

An important aspect of ensuring that the template can be reused across a range of inputs is input validation. The following code is an example of verifying that the stopping fidelity during AQC-Tensor has been specified appropriately and if not, returning an informative error message for how to fix the error.

In [5]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

# Perform parameter validation

if not 0.0 < aqc_stopping_fidelity <= 1.0:
    raise ValueError(
        f"Invalid stopping fidelity: {aqc_stopping_fidelity}.  It must be a positive float no greater than 1."
    )

Appending to ./source_files/template_hamiltonian_simulation.py


Quando o template de função está em execução, é útil retornar informações nos logs usando declarações print, para que você possa avaliar melhor o progresso da carga de trabalho. A seguir, há um exemplo simples de impressão das `estimator_options`, para que haja um registro das opções reais do Estimator utilizadas. Há muitos outros exemplos semelhantes ao longo do programa para reportar o progresso durante a execução, incluindo o valor da função objetivo durante o componente iterativo do AQC-Tensor e a profundidade de dois qubits do circuito da arquitetura de conjunto de instruções (ISA) final destinado à execução em hardware.

In [6]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

output = {}

Appending to ./source_files/template_hamiltonian_simulation.py


### Map the problem and pre-process the circuit with AQC

The AQC-Tensor optimization happens in step 1 of a Qiskit pattern.  First, a target state is constructed.  In this example, it is constructed from a target circuit that evolves the same Hamiltonian for the same time period as the AQC portion.  Then, an ansatz is generated from an equivalent circuit but with fewer Trotter steps.  In the main portion of the AQC algorithm, that ansatz is iteratively brought closer to the target state.  Finally, the result is combined with the remainder of the Trotter steps needed to reach the desired evolution time.

Note the additional examples of logging incorporated in the following code.

In [7]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

import os
os.environ["NUMBA_CACHE_DIR"] = "/data"

import datetime
import quimb.tensor
from scipy.optimize import OptimizeResult, minimize
from qiskit.synthesis import SuzukiTrotter
from qiskit_addon_utils.problem_generators import generate_time_evolution_circuit
from qiskit_addon_aqc_tensor.ansatz_generation import (
    generate_ansatz_from_circuit,
    AnsatzBlock,
)
from qiskit_addon_aqc_tensor.simulation import (
    tensornetwork_from_circuit,
    compute_overlap,
)
from qiskit_addon_aqc_tensor.simulation.quimb import QuimbSimulator
from qiskit_addon_aqc_tensor.objective import OneMinusFidelity

print("Hamiltonian:", hamiltonian)
print("Observable:", observable)
simulator_settings = QuimbSimulator(quimb.tensor.CircuitMPS, autodiff_backend="jax")

# Construct the AQC target circuit
aqc_target_circuit = initial_state.copy()
if aqc_evolution_time:
    aqc_target_circuit.compose(
        generate_time_evolution_circuit(
            hamiltonian,
            synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
            time=aqc_evolution_time,
        ),
        inplace=True,
    )

# Construct matrix-product state representation of the AQC target state
aqc_target_mps = tensornetwork_from_circuit(aqc_target_circuit, simulator_settings)
print("Target MPS maximum bond dimension:", aqc_target_mps.psi.max_bond())
output["target_bond_dimension"] = aqc_target_mps.psi.max_bond()

# Generate an ansatz and initial parameters from a Trotter circuit with fewer steps
aqc_good_circuit = initial_state.copy()
if aqc_evolution_time:
    aqc_good_circuit.compose(
        generate_time_evolution_circuit(
            hamiltonian,
            synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
            time=aqc_evolution_time,
        ),
        inplace=True,
    )
aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(aqc_good_circuit)
print("Number of AQC parameters:", len(aqc_initial_parameters))
output["num_aqc_parameters"] = len(aqc_initial_parameters)

# Calculate the fidelity of ansatz circuit vs. the target state, before optimization
good_mps = tensornetwork_from_circuit(aqc_good_circuit, simulator_settings)
starting_fidelity = abs(compute_overlap(good_mps, aqc_target_mps)) ** 2
print("Starting fidelity of AQC portion:", starting_fidelity)
output["aqc_starting_fidelity"] = starting_fidelity

# Optimize the ansatz parameters by using MPS calculations
def callback(intermediate_result: OptimizeResult):
    fidelity = 1 - intermediate_result.fun
    print(f"{datetime.datetime.now()} Intermediate result: Fidelity {fidelity:.8f}")
    if intermediate_result.fun < stopping_point:
        raise StopIteration


objective = OneMinusFidelity(aqc_target_mps, aqc_ansatz, simulator_settings)
stopping_point = 1.0 - aqc_stopping_fidelity

result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": aqc_max_iterations},
    callback=callback,
)
if result.status not in (
    0,
    1,
    99,
):  # 0 => success; 1 => max iterations reached; 99 => early termination via StopIteration
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )
print(f"Done after {result.nit} iterations.")
output["num_iterations"] = result.nit
aqc_final_parameters = result.x
output["aqc_final_parameters"] = list(aqc_final_parameters)

# Construct an optimized circuit for initial portion of time evolution
aqc_final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)

# Calculate fidelity after optimization
aqc_final_mps = tensornetwork_from_circuit(aqc_final_circuit, simulator_settings)
aqc_fidelity = abs(compute_overlap(aqc_final_mps, aqc_target_mps)) ** 2
print("Fidelity of AQC portion:", aqc_fidelity)
output["aqc_fidelity"] = aqc_fidelity

# Construct final circuit, with remainder of time evolution
final_circuit = aqc_final_circuit.copy()
if remainder_evolution_time:
    remainder_circuit = generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=remainder_num_trotter_steps),
        time=remainder_evolution_time,
    )
    final_circuit.compose(remainder_circuit, inplace=True)

Appending to ./source_files/template_hamiltonian_simulation.py


#### Validar as entradas
Um aspecto importante para garantir que o template possa ser reutilizado em um conjunto de entradas é a validação de entradas. O código a seguir é um exemplo de verificação de que a fidelidade de parada durante o AQC-Tensor foi especificada adequadamente e, caso contrário, retorna uma mensagem de erro informativa sobre como corrigir o erro.

In [8]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.backend(backend_name)

# Transpile PUBs (circuits and observables) to match ISA
pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuit = pass_manager.run(final_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

isa_2qubit_depth = isa_circuit.depth(lambda x: x.operation.num_qubits == 2)
print("ISA circuit two-qubit depth:", isa_2qubit_depth)
output["twoqubit_depth"] = isa_2qubit_depth

Appending to ./source_files/template_hamiltonian_simulation.py


#### Exit early if using dry run mode

If dry run mode has been selected, then the program is stopped before executing on hardware. This can be useful if, for example, you want first to inspect the two-qubit depth of the ISA circuit before deciding to execute on hardware.

In [9]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

# Exit now if dry run; don't execute on hardware
if dry_run:
    import sys

    print("Exiting before hardware execution since `dry_run` is True.")
    save_result(output)
    sys.exit(0)

Appending to ./source_files/template_hamiltonian_simulation.py


#### Preparar as saídas da função
Primeiro, prepare um dicionário para armazenar todas as saídas do template de função. As chaves serão adicionadas a este dicionário ao longo do fluxo de trabalho, e ele é retornado ao final do programa.

In [10]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

# ## Step 3: Execute quantum experiments on backend
from qiskit_ibm_runtime import EstimatorV2 as Estimator


estimator = Estimator(backend, options=estimator_options)

# Submit the underlying Estimator job. Note that this is not the
# actual function job.
job = estimator.run([(isa_circuit, isa_observable)])
print("Job ID:", job.job_id())
output["job_id"] = job.job_id()

# Wait until job is complete
hw_results = job.result()
hw_results_dicts = [pub_result.data.__dict__ for pub_result in hw_results]

# Save hardware results to serverless output dictionary
output["hw_results"] = hw_results_dicts

# Reorganize expectation values
hw_expvals = [pub_result_data["evs"].tolist() for pub_result_data in hw_results_dicts]

# Save expectation values to Qiskit Serverless
print("Hardware expectation values", hw_expvals)
output["hw_expvals"] = hw_expvals[0]

Appending to ./source_files/template_hamiltonian_simulation.py


#### Save the output

This function template returns the relevant domain-level output for this Hamiltonian simulation workflow (expectation values) in addition to important metadata generated along the way.

In [11]:
%%writefile --append ./source_files/template_hamiltonian_simulation.py

save_result(output)

Appending to ./source_files/template_hamiltonian_simulation.py


### Mapear o problema e pré-processar o circuito com AQC
A otimização AQC-Tensor ocorre na etapa 1 de um padrão Qiskit. Primeiro, um estado alvo é construído. Neste exemplo, ele é construído a partir de um circuito alvo que evolui o mesmo Hamiltoniano pelo mesmo período de tempo que a parte AQC. Em seguida, um ansatz é gerado a partir de um circuito equivalente, mas com menos etapas de Trotter. Na parte principal do algoritmo AQC, esse ansatz é iterativamente aproximado ao estado alvo. Por fim, o resultado é combinado com o restante das etapas de Trotter necessárias para atingir o tempo de evolução desejado.

Observe os exemplos adicionais de registro incorporados no código a seguir.

In [12]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

This program has custom `pip` dependencies.  Add them to a `dependencies` array when constructing the `QiskitFunction` instance:

In [13]:
template = QiskitFunction(
    title="template_hamiltonian_simulation",
    entrypoint="template_hamiltonian_simulation.py",
    working_dir="./source_files/",
    dependencies=[
        "qiskit-addon-utils~=0.1.0",
        "qiskit-addon-aqc-tensor[quimb-jax]~=0.1.2",
        "mergedeep==1.3.4",
    ],
)

In [14]:
serverless.upload(template)

QiskitFunction(template_hamiltonian_simulation)

Finally, check if the program successfully uploaded, use `serverless.list()`:

In [15]:
serverless.list()

 QiskitFunction(template_hamiltonian_simulation),


#### Sair antecipadamente se estiver usando o modo de execução a seco
Se o modo de execução a seco (dry run) foi selecionado, então o programa é interrompido antes de executar no hardware. Isso pode ser útil se, por exemplo, você quiser primeiro inspecionar a profundidade de dois qubits do circuito ISA antes de decidir executar no hardware.

In [16]:
template = serverless.load("template_hamiltonian_simulation")

Next, run the template with the domain-level inputs for Hamiltonian simulation. This example specifies a 50-qubit XXZ model with random couplings, and an initial state and observable.

In [17]:
from itertools import chain
import numpy as np
from qiskit.quantum_info import SparsePauliOp

L = 50

# Generate the edge list for this spin-chain
edges = [(i, i + 1) for i in range(L - 1)]
# Generate an edge-coloring so we can make hw-efficient circuits
edges = edges[::2] + edges[1::2]

# Generate random coefficients for our XXZ Hamiltonian
np.random.seed(0)
Js = np.random.rand(L - 1) + 0.5 * np.ones(L - 1)

hamiltonian = SparsePauliOp.from_sparse_list(
    chain.from_iterable(
        [
            [
                ("XX", (i, j), Js[i] / 2),
                ("YY", (i, j), Js[i] / 2),
                ("ZZ", (i, j), Js[i]),
            ]
            for i, j in edges
        ]
    ),
    num_qubits=L,
)
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

In [18]:
from qiskit import QuantumCircuit

initial_state = QuantumCircuit(L)
for i in range(L):
    if i % 2:
        initial_state.x(i)

In [19]:
job = template.run(
    dry_run=True,
    initial_state=initial_state,
    hamiltonian=hamiltonian,
    observable=observable,
    backend_name="ibm_fez",
    estimator_options={},
    aqc_evolution_time=0.2,
    aqc_ansatz_num_trotter_steps=1,
    aqc_target_num_trotter_steps=32,
    remainder_evolution_time=0.2,
    remainder_num_trotter_steps=4,
    aqc_max_iterations=300,
)
print(job.job_id)

853b0edb-d63f-4629-be71-398b6dcf33cb


#### Salvar a saída
Este template de função retorna a saída relevante em nível de domínio para este fluxo de trabalho de simulação hamiltoniana (valores esperados), além de metadados importantes gerados ao longo do processo.

In [20]:
job.status()

'QUEUED'

After the job is running, you can fetch logs created from the `print()` outputs. These can provide actionable information about the progress of the Hamiltonian simulation workflow. For example, the value of the objective function during the iterative component of AQC, or the two-qubit depth of the final ISA circuit intended for execution on hardware.

In [21]:
print(job.logs())

No logs yet.


## Implantar a função na IBM Quantum Platform
A seção anterior criou um programa para ser executado remotamente. O código nesta seção faz o upload desse programa para o Qiskit Serverless.

Use `qiskit-ibm-catalog` para autenticar no `QiskitServerless` com sua chave de API, que você pode encontrar no painel da [IBM Quantum Platform](https://quantum.cloud.ibm.com), e fazer o upload do programa.

Você pode opcionalmente usar `save_account()` para salvar suas credenciais (consulte o guia [Configurar sua conta IBM Cloud](/guides/cloud-setup#cloud-save)). Observe que isso grava suas credenciais no mesmo arquivo que [`QiskitRuntimeService.save_account()`.](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/qiskit-runtime-service#save_account)

In [22]:
result = job.result()

del result[
    "aqc_final_parameters"
]  # the list is too long to conveniently display here
result

{'target_bond_dimension': 5,
 'num_aqc_parameters': 816,
 'aqc_starting_fidelity': 0.9914382555614002,
 'num_iterations': 72,
 'aqc_fidelity': 0.9998108844412502,
 'twoqubit_depth': 33}

Este programa tem dependências `pip` personalizadas. Adicione-as a um array `dependencies` ao construir a instância `QiskitFunction`:

In [23]:
print(job.logs())

2024-12-17 14:50:15,580	INFO job_manager.py:531 -- Runtime env is setting up.
estimator_options = {
    "resilience": {
        "measure_mitigation": true,
        "zne_mitigation": true,
        "zne": {
            "amplifier": "gate_folding",
            "noise_factors": [
                1,
                2,
                3
            ],
            "extrapolated_noise_factors": [
                0.0,
                0.1,
                0.2,
                0.30000000000000004,
                0.4,
                0.5,
                0.6000000000000001,
                0.7000000000000001,
                0.8,
                0.9,
                1.0,
                1.1,
                1.2000000000000002,
                1.3,
                1.4000000000000001,
                1.5,
                1.6,
                1.7000000000000002,
                1.8,
                1.9000000000000001,
                2.0,
                2.1,
                2.2,
                2.3

## Next steps

<Admonition type="info" title="Recommendations">

For a deeper dive into the AQC-Tensor Qiskit addon, check out the [Improved Trotterized Time Evolution with Approximate Quantum Compilation](/docs/tutorials/approximate-quantum-compilation-for-time-evolution) tutorial or the [qiskit-addon-aqc-tensor repository](https://github.com/Qiskit/qiskit-addon-aqc-tensor).

</Admonition>

In [ ]:
%%writefile ./source_files/template_hamiltonian_simulation_full.py

from qiskit import QuantumCircuit
from qiskit_serverless import get_arguments, save_result


# Extract parameters from arguments
#
# Do this at the top of the program so it fails early if any required arguments are missing or invalid.

arguments = get_arguments()

dry_run = arguments.get("dry_run", False)
backend_name = arguments["backend_name"]

aqc_evolution_time = arguments["aqc_evolution_time"]
aqc_ansatz_num_trotter_steps = arguments["aqc_ansatz_num_trotter_steps"]
aqc_target_num_trotter_steps = arguments["aqc_target_num_trotter_steps"]

remainder_evolution_time = arguments["remainder_evolution_time"]
remainder_num_trotter_steps = arguments["remainder_num_trotter_steps"]

# Stop if this fidelity is achieved
aqc_stopping_fidelity = arguments.get("aqc_stopping_fidelity", 1.0)
# Stop after this number of iterations, even if stopping fidelity is not achieved
aqc_max_iterations = arguments.get("aqc_max_iterations", 500)

hamiltonian = arguments["hamiltonian"]
observable = arguments["observable"]
initial_state = arguments.get("initial_state", QuantumCircuit(hamiltonian.num_qubits))

import numpy as np
import json
from mergedeep import merge


# Configure `EstimatorOptions`, to control the parameters of the hardware experiment
#
# Set default options
estimator_default_options = {
    "resilience": {
        "measure_mitigation": True,
        "zne_mitigation": True,
        "zne": {
            "amplifier": "gate_folding",
            "noise_factors": [1, 2, 3],
            "extrapolated_noise_factors": list(np.linspace(0, 3, 31)),
            "extrapolator": ["exponential", "linear", "fallback"],
        },
        "measure_noise_learning": {
            "num_randomizations": 512,
            "shots_per_randomization": 512,
        },
    },
    "twirling": {
        "enable_gates": True,
        "enable_measure": True,
        "num_randomizations": 300,
        "shots_per_randomization": 100,
        "strategy": "active",
    },
}
# Merge with user-provided options
estimator_options = merge(
    arguments.get("estimator_options", {}), estimator_default_options
)

print("estimator_options =", json.dumps(estimator_options, indent=4))

# Perform parameter validation

if not 0.0 < aqc_stopping_fidelity <= 1.0:
    raise ValueError(
        f"Invalid stopping fidelity: {aqc_stopping_fidelity}.  It must be a positive float no greater than 1."
    )

output = {}

import os
os.environ["NUMBA_CACHE_DIR"] = "/data"

import datetime
import quimb.tensor
from scipy.optimize import OptimizeResult, minimize
from qiskit.synthesis import SuzukiTrotter
from qiskit_addon_utils.problem_generators import generate_time_evolution_circuit
from qiskit_addon_aqc_tensor.ansatz_generation import (
    generate_ansatz_from_circuit,
    AnsatzBlock,
)
from qiskit_addon_aqc_tensor.simulation import (
    tensornetwork_from_circuit,
    compute_overlap,
)
from qiskit_addon_aqc_tensor.simulation.quimb import QuimbSimulator
from qiskit_addon_aqc_tensor.objective import OneMinusFidelity

print("Hamiltonian:", hamiltonian)
print("Observable:", observable)
simulator_settings = QuimbSimulator(quimb.tensor.CircuitMPS, autodiff_backend="jax")

# Construct the AQC target circuit
aqc_target_circuit = initial_state.copy()
if aqc_evolution_time:
    aqc_target_circuit.compose(
        generate_time_evolution_circuit(
            hamiltonian,
            synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
            time=aqc_evolution_time,
        ),
        inplace=True,
    )

# Construct matrix-product state representation of the AQC target state
aqc_target_mps = tensornetwork_from_circuit(aqc_target_circuit, simulator_settings)
print("Target MPS maximum bond dimension:", aqc_target_mps.psi.max_bond())
output["target_bond_dimension"] = aqc_target_mps.psi.max_bond()

# Generate an ansatz and initial parameters from a Trotter circuit with fewer steps
aqc_good_circuit = initial_state.copy()
if aqc_evolution_time:
    aqc_good_circuit.compose(
        generate_time_evolution_circuit(
            hamiltonian,
            synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
            time=aqc_evolution_time,
        ),
        inplace=True,
    )
aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(aqc_good_circuit)
print("Number of AQC parameters:", len(aqc_initial_parameters))
output["num_aqc_parameters"] = len(aqc_initial_parameters)

# Calculate the fidelity of ansatz circuit vs. the target state, before optimization
good_mps = tensornetwork_from_circuit(aqc_good_circuit, simulator_settings)
starting_fidelity = abs(compute_overlap(good_mps, aqc_target_mps)) ** 2
print("Starting fidelity of AQC portion:", starting_fidelity)
output["aqc_starting_fidelity"] = starting_fidelity

# Optimize the ansatz parameters by using MPS calculations
def callback(intermediate_result: OptimizeResult):
    fidelity = 1 - intermediate_result.fun
    print(f"{datetime.datetime.now()} Intermediate result: Fidelity {fidelity:.8f}")
    if intermediate_result.fun < stopping_point:
        raise StopIteration


objective = OneMinusFidelity(aqc_target_mps, aqc_ansatz, simulator_settings)
stopping_point = 1.0 - aqc_stopping_fidelity

result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": aqc_max_iterations},
    callback=callback,
)
if result.status not in (
    0,
    1,
    99,
):  # 0 => success; 1 => max iterations reached; 99 => early termination via StopIteration
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )
print(f"Done after {result.nit} iterations.")
output["num_iterations"] = result.nit
aqc_final_parameters = result.x
output["aqc_final_parameters"] = list(aqc_final_parameters)

# Construct an optimized circuit for initial portion of time evolution
aqc_final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)

# Calculate fidelity after optimization
aqc_final_mps = tensornetwork_from_circuit(aqc_final_circuit, simulator_settings)
aqc_fidelity = abs(compute_overlap(aqc_final_mps, aqc_target_mps)) ** 2
print("Fidelity of AQC portion:", aqc_fidelity)
output["aqc_fidelity"] = aqc_fidelity

# Construct final circuit, with remainder of time evolution
final_circuit = aqc_final_circuit.copy()
if remainder_evolution_time:
    remainder_circuit = generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=remainder_num_trotter_steps),
        time=remainder_evolution_time,
    )
    final_circuit.compose(remainder_circuit, inplace=True)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.backend(backend_name)

# Transpile PUBs (circuits and observables) to match ISA
pass_manager = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuit = pass_manager.run(final_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)

isa_2qubit_depth = isa_circuit.depth(lambda x: x.operation.num_qubits == 2)
print("ISA circuit two-qubit depth:", isa_2qubit_depth)
output["twoqubit_depth"] = isa_2qubit_depth

# Exit now if dry run; don't execute on hardware
if dry_run:
    import sys

    print("Exiting before hardware execution since `dry_run` is True.")
    save_result(output)
    sys.exit(0)

# ## Step 3: Execute quantum experiments on backend
from qiskit_ibm_runtime import EstimatorV2 as Estimator


estimator = Estimator(backend, options=estimator_options)

# Submit the underlying Estimator job. Note that this is not the
# actual function job.
job = estimator.run([(isa_circuit, isa_observable)])
print("Job ID:", job.job_id())
output["job_id"] = job.job_id()

# Wait until job is complete
hw_results = job.result()
hw_results_dicts = [pub_result.data.__dict__ for pub_result in hw_results]

# Save hardware results to serverless output dictionary
output["hw_results"] = hw_results_dicts

# Reorganize expectation values
hw_expvals = [pub_result_data["evs"].tolist() for pub_result_data in hw_results_dicts]

# Save expectation values to Qiskit Serverless
output["hw_expvals"] = hw_expvals[0]

save_result(output)

Overwriting ./source_files/template_hamiltonian_simulation_full.py


<details>
<summary><b>Full program source code</b></summary>

Here is the entire source of `./source_files/template_hamiltonian_simulation.py` as one code block.

<CodeCellPlaceholder tag="id-full-source" />

</details>

In [24]:
# This cell is hidden from users.  It verifies both source listings are identical then deletes the working folder we created
import shutil

with open("./source_files/template_hamiltonian_simulation.py") as f1:
    with open("./source_files/template_hamiltonian_simulation_full.py") as f2:
        assert f1.read() == f2.read()

shutil.rmtree("./source_files/")